# Concept ↔ image-region explanations: Qwen3-VL attention, CLIP Surgery, Grad-ECLIP

**Experiment v1 — Kaggle, 2 GPUs.** Reads images/captions from [`zintom/blackbox-adversarial-attack-dataset`](https://www.kaggle.com/datasets/zintom/blackbox-adversarial-attack-dataset), asks **Qwen3-VL-2B-Instruct** to identify image-grounded concepts, and creates a synchronized heatmap panel for **each concept and the full caption**. An optional figure shows attention for the individual Qwen subword tokens in the full caption.

### Scientific interpretation

| Method | What it measures | Is it target-specific? | Is it gradient-based? |
| --- | --- | --- | --- |
| Qwen3-VL causal attention | Attention **from prompt tokens naming the concept** to earlier image tokens, averaged over chosen decoder layers/heads | Query phrase-specific | **No** |
| CLIP Surgery (official implementation) | Text-conditioned local similarity map, with architecture + feature surgery | Yes | **No** |
| Grad-ECLIP (paper + author notebook, single-head last-layer formulation) | Local contribution to CLIP image–text cosine similarity, using a gradient-weighted value and relaxed QK spatial weighting | Yes | **Yes** |

**Caution:** The three maps are not calibrated probabilities and their numerical magnitudes are not comparable. Qwen prompt-token attention is neither a generation-time attention explanation nor proof of causal image-region importance. *Leave-one-concept-out* text scores and *deletion* tests below are complementary interventions, not ground-truth causal explanations. Qwen's caption/concepts are generated hypotheses; independently check hallucinations.

**Design choice:** For a fair spatial overlay, one input image is supplied to each model without manual crop. The Qwen processor preserves aspect ratio and dynamically resizes; CLIP uses a resized square (a model-specific geometric difference). Images are visualized on original aspect ratio. CLIP comparisons share the original ViT-B/16 weights, but Surgery changes its inference path.

**Implementation provenance:** [CLIP Surgery paper/code](https://github.com/xmed-lab/CLIP_Surgery); [Grad-ECLIP paper](https://proceedings.mlr.press/v235/zhao24p.html) and [author's Grad-ECLIP notebook](https://github.com/Cyang-Zhao/Grad-Eclip/blob/main/grad_eclip_image.ipynb); [Qwen3-VL source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_vl/modeling_qwen3_vl.py). This Grad-ECLIP implementation reproduces the author's last-layer, **single-head reconstruction**, rather than simply calling vanilla Grad-CAM.

### Kaggle configuration

1. Set **Accelerator → GPU T4 ×2** (or any Kaggle 2-GPU option) and **Internet ON** for first-time model/repository downloads. Add your dataset as **Notebook Input**. Alternatively, the optional `kagglehub` downloader will try to fetch it.
2. Start with `MAX_IMAGES=2`, `MAX_CONCEPTS=4`; increase after confirming memory/time. `QWEN_MAX_PIXELS` controls attention sequence cost.
3. GPU 0 hosts Qwen3-VL; GPU 1 hosts two ViT-B/16 model instances. CLIP computation is **pipelined with Qwen on the next image** using one worker thread and a bounded queue; this uses both GPUs concurrently after the first sample.
4. Figures and CSV/JSON/NPZ go under `/kaggle/working/concept_grad_v1/`. Outputs are not precomputed in this notebook; running the cells is required.

In [ ]:
# Install once with Kaggle Internet enabled. Restart kernel only if Kaggle reports a package conflict.
%pip -q install -U "transformers>=4.57.1,<5" "accelerate>=1.6" "ftfy" "regex" "kagglehub" "nbformat" "pandas" "matplotlib" "opencv-python-headless" "safetensors" "setuptools<81"
%pip -q install git+https://github.com/openai/CLIP.git

In [ ]:
import os, sys, gc, re, json, math, time, copy, random, subprocess, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()
assert torch.cuda.device_count() >= 2, 'This notebook needs 2 accessible GPUs. Set Kaggle Accelerator to GPU T4 x2.'
QWEN_DEVICE = torch.device('cuda:0')
CLIP_DEVICE = torch.device('cuda:1')
OUTPUT_DIR = Path('/kaggle/working/concept_grad_v1' if Path('/kaggle/working').exists() else '/mnt/data/concept_grad_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment controls
MAX_IMAGES = 2                    # Smoke test first; set to None for full dataset
MAX_CONCEPTS = 4                  # per image; full caption always included
MAX_QWEN_TOKENS = 180             # for initial LLM caption/concept extraction only
QWEN_MAX_PIXELS = 448 * 448      # reduce to 384**2 if T4 OOM; processor rounds to its patch constraints
QWEN_MIN_PIXELS = 256 * 256
QWEN_LAYER_FRACTIONS = (0.30, 0.55, 0.80, 0.96)
CLIP_SIDE = 224                 # official ViT-B/16 benchmark setup; 14x14 patch grid
TOP_MASK_FRACTION = 0.20       # deletion-test important-area fraction
RUN_DELETION = True
VISUALIZE_CAPTION_SUBTOKENS = True
SAVE_NPZ = True
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
DATASET_SLUG = 'zintom/blackbox-adversarial-attack-dataset'
DATASET_DIR = None             # set e.g. '/kaggle/input/blackbox-adversarial-attack-dataset'
CAPTION_OVERRIDE = None        # optional caption applied to every image (usually leave None)

print('GPUs:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
print('PyTorch:', torch.__version__, '| outputs:', OUTPUT_DIR)

## 1. Discover image files and optional captions

The Kaggle dataset webpage does not expose a reliable machine-readable file listing to this notebook author. **No folder names or annotation columns are assumed.** This cell prints the exact mounted structure and attempts to resolve JSON/JSONL/CSV/Parquet caption records, including COCO-style `images`/`annotations`. Check `preview` before running inference. It does not silently substitute another dataset.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
TEXT_EXTS = {'.csv', '.json', '.jsonl', '.parquet'}
CAPTION_KEYS = ('caption', 'captions', 'description', 'text', 'sentence')
IMAGE_KEYS = ('file_name', 'filename', 'image_path', 'path', 'image', 'img', 'file')
ID_KEYS = ('image_id', 'id')

def discover_root():
    if DATASET_DIR is not None:
        root = Path(DATASET_DIR)
        if not root.exists(): raise FileNotFoundError(root)
        return root
    base = Path('/kaggle/input')
    dirs = [p for p in base.glob('*') if p.is_dir()] if base.exists() else []
    exact = next((p for p in dirs if p.name == 'blackbox-adversarial-attack-dataset'), None)
    if exact: return exact
    matches = [p for p in dirs if 'blackbox' in p.name.lower() and 'attack' in p.name.lower()]
    if len(matches) == 1: return matches[0]
    print('Mounted Kaggle input folders:', [p.name for p in dirs])
    import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    return root

ROOT = discover_root()
print('DATASET ROOT:', ROOT)
files = sorted((p for p in ROOT.rglob('*') if p.is_file()), key=lambda p: str(p))
print('Total files:', len(files))
print('First files:', '\n'.join('  ' + str(p.relative_to(ROOT)) for p in files[:30]))
image_files = [p for p in files if p.suffix.lower() in IMAGE_EXTS]
meta_files = [p for p in files if p.suffix.lower() in TEXT_EXTS]
print('Images:', len(image_files), '| metadata candidates:', len(meta_files))
assert image_files, 'No ordinary image files found. Inspect archive formats / set DATASET_DIR appropriately.'

# Map by basename/stem/relative path, resolving ambiguous matches conservatively.
from collections import defaultdict
lookup = defaultdict(list)
for p in image_files:
    for k in {p.name.lower(), p.stem.lower(), p.relative_to(ROOT).as_posix().lower()}:
        lookup[k].append(p)

def resolve_image(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).replace('\\', '/').strip().lower().lstrip('./')
    candidates = lookup.get(s, []) or lookup.get(Path(s).name, []) or lookup.get(Path(s).stem, [])
    return str(candidates[0]) if len(candidates) == 1 else None

def get_first(rec, keys):
    for k in keys:
        if k in rec and rec[k] is not None and not isinstance(rec[k], (list, dict)):
            return rec[k]
    return None

def records_from_metadata(path):
    if path.stat().st_size > 120 * 1024 * 1024: return []
    try:
        if path.suffix.lower() == '.csv':
            return pd.read_csv(path, nrows=100000).to_dict('records')
        if path.suffix.lower() == '.parquet':
            return pd.read_parquet(path).head(100000).to_dict('records')
        if path.suffix.lower() == '.jsonl':
            with path.open(encoding='utf-8') as f: return [json.loads(line) for _, line in zip(range(100000), f) if line.strip()]
        obj = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(obj, list): return obj
        if isinstance(obj, dict):
            if 'annotations' in obj and 'images' in obj:
                names = {str(r.get('id')): get_first(r, IMAGE_KEYS) for r in obj['images'] if isinstance(r, dict)}
                return [dict(image=names.get(str(a.get('image_id'))), caption=get_first(a, CAPTION_KEYS))
                        for a in obj['annotations'] if isinstance(a, dict)]
            for key in ('data', 'records', 'items', 'samples'):
                if isinstance(obj.get(key), list): return obj[key]
    except Exception as e:
        print('Skipping metadata', path.name, type(e).__name__, str(e)[:100])
    return []

caption_map = {}
for mf in meta_files[:80]:
    for r in records_from_metadata(mf):
        if not isinstance(r, dict): continue
        cap = get_first(r, CAPTION_KEYS)
        direct = resolve_image(get_first(r, IMAGE_KEYS))
        if direct is None and get_first(r, ID_KEYS) is not None:
            direct = resolve_image(str(get_first(r, ID_KEYS)))
        if direct and isinstance(cap, str) and cap.strip() and direct not in caption_map:
            caption_map[direct] = cap.strip()
print('Images with matched captions:', len(caption_map))

samples = [{'image_id': p.stem, 'image_path': str(p),
            'reference_caption': CAPTION_OVERRIDE or caption_map.get(str(p))}
           for p in image_files]
if MAX_IMAGES is not None:
    samples = samples[:MAX_IMAGES]
preview = pd.DataFrame(samples)
display(preview.head(10))
print('Selected:', len(samples))

## 2. Load Qwen on GPU 0 and two official CLIP models on GPU 1

The Surgery repository exports its own `clip` package. Import it under a distinct package name so it cannot shadow the stock OpenAI `clip` used by Grad-ECLIP. The Surgery checkpoint's architecture change is invoked via `CS-ViT-B/16`, *not* reproduced by a generic CLIP cosine map.

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, __version__ as HF_VERSION
import clip as openai_clip
import importlib.util

SURGERY_SRC = OUTPUT_DIR / '_CLIP_Surgery'
if not (SURGERY_SRC / 'clip' / '__init__.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/xmed-lab/CLIP_Surgery.git', str(SURGERY_SRC)], check=True)
spec = importlib.util.spec_from_file_location('surgery_clip', SURGERY_SRC / 'clip' / '__init__.py',
                                              submodule_search_locations=[str(SURGERY_SRC / 'clip')])
surgery_clip = importlib.util.module_from_spec(spec)
sys.modules['surgery_clip'] = surgery_clip
spec.loader.exec_module(surgery_clip)

QWEN_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print('transformers:', HF_VERSION, '| Qwen dtype:', QWEN_DTYPE)
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
# Official Qwen3-VL pixel budget: size longest_edge=max_pixels, shortest_edge=min_pixels.
qwen_processor.image_processor.size = {'longest_edge': QWEN_MAX_PIXELS, 'shortest_edge': QWEN_MIN_PIXELS}
qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, torch_dtype=QWEN_DTYPE, attn_implementation='sdpa',
    low_cpu_mem_usage=True,
).to(QWEN_DEVICE).eval()
for p in qwen.parameters(): p.requires_grad_(False)

surgery_model, _ = surgery_clip.load('CS-ViT-B/16', device=str(CLIP_DEVICE), jit=False)
surgery_model.eval()
for p in surgery_model.parameters(): p.requires_grad_(False)
clip_model, _ = openai_clip.load('ViT-B/16', device=str(CLIP_DEVICE), jit=False)
clip_model.eval()
for p in clip_model.parameters(): p.requires_grad_(False)
print('Models ready | qwen GPU:', next(qwen.parameters()).device,
      '| surgery GPU:', next(surgery_model.parameters()).device,
      '| vanilla CLIP GPU:', next(clip_model.parameters()).device)

## 3. LLM: image-grounded caption and concepts

Existing dataset captions are kept as `reference_caption`; Qwen produces a short set of **visually grounded** noun phrases, actions or relations. Without a reference caption, Qwen also generates a caption. Parsing is strict enough to expose malformed responses rather than silently inventing labels. This stage is only a proposal mechanism, not labeled ground truth.

In [ ]:
CONCEPT_SYSTEM = ('You are a careful visual annotator. Respond with one valid JSON object only. '
                  'Do not use markdown fences. Do not claim objects or actions absent from the image.')

def qwen_inputs(image, text):
    msgs = [{'role': 'system', 'content': [{'type': 'text', 'text': CONCEPT_SYSTEM}]},
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': text}]}]
    chat = qwen_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # max_pixels/min_pixels are processor-specific in some versions; use image processor size explicitly.
    return qwen_processor(text=[chat], images=[image], padding=False, return_tensors='pt').to(QWEN_DEVICE)

def load_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

def robust_json(text):
    candidate = re.search(r'\{[\s\S]*\}', text)
    if not candidate:
        raise ValueError('No JSON in Qwen output: ' + text[:300])
    obj = json.loads(candidate.group(0))
    if not isinstance(obj, dict) or not isinstance(obj.get('concepts'), list):
        raise ValueError('Expected JSON with concepts list: ' + text[:300])
    return obj

@torch.inference_mode()
def extract_caption_concepts(image, reference_caption=None):
    if reference_caption:
        instr = (f'Reference caption: {reference_caption}\n'
                 'Return JSON: {"caption": "reference caption unchanged", '
                 '"concepts": ["short visible noun phrase", "visible action or relation"]}. '
                 'Keep the caption identical to the reference. Extract at most 6 distinct concepts relevant '
                 'to this caption and supported by the image; avoid generic words.')
    else:
        instr = ('Describe the image faithfully in one concise English sentence and extract at most 6 distinct, '
                 'specific visual noun phrases/actions/relations. Return JSON exactly with keys '
                 '{"caption": "...", "concepts": ["...", "..."]}. Avoid guesses.')
    enc = qwen_inputs(image, instr)
    output = qwen.generate(**enc, max_new_tokens=MAX_QWEN_TOKENS, do_sample=False,
                           use_cache=True, pad_token_id=qwen_processor.tokenizer.pad_token_id)
    generated = output[:, enc['input_ids'].shape[1]:]
    raw = qwen_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    obj = robust_json(raw)
    caption = str(reference_caption or obj.get('caption', '')).strip()
    if not caption: raise ValueError('Qwen returned empty caption: ' + raw[:300])
    concepts = []
    for c in obj['concepts']:
        if not isinstance(c, str): continue
        c = re.sub(r'\s+', ' ', c).strip(' .,"\'')
        if c and len(c) <= 100 and c.casefold() != caption.casefold() and c.casefold() not in [x.casefold() for x in concepts]:
            concepts.append(c)
    if not concepts: raise ValueError('No usable grounded concepts. Raw model output: ' + raw[:300])
    return caption, concepts[:MAX_CONCEPTS], raw

# Optional smoke test after model downloads:
# im = load_rgb(samples[0]['image_path'])
# extract_caption_concepts(im, samples[0]['reference_caption'])

## 4. Qwen3-VL attention: exact query-token → image-token probabilities

Instead of `output_attentions=True` on **all** Qwen layers (which would materialize expensive `L×L` matrices), a **pre-forward hook** reconstructs the selected Q/K projections **after Q/K normalization and rotary position encoding**, using the same formula as Qwen3-VL attention. It calculates only rows for the concept-query tokens; the rest of Qwen's normal SDPA forward remains untouched. Important: softmax normalization is over **all causally available tokens**, not just image tokens. The image-token mass is logged separately and the heatmap is normalized **only for display**.

The query itself appears *after* the image inside the user prompt, so its causal attention may reach the image. Token spans are found by comparing tokenized prompts with/without the trailing phrase, rather than guessing character offsets or assuming every word is one token. An assertion checks the image patch grid against the actual `input_ids` image-token positions.

In [ ]:
from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb
from contextlib import contextmanager

QWEN_NLAYERS = len(qwen.model.language_model.layers)
QWEN_LAYERS = sorted(set(min(QWEN_NLAYERS - 1, max(0, round((QWEN_NLAYERS - 1) * f)))
                         for f in QWEN_LAYER_FRACTIONS))
assert QWEN_LAYERS, 'No Qwen attention layers selected'
print('Selected Qwen language-model layers:', QWEN_LAYERS, '/', QWEN_NLAYERS)

# Minimal phrase query; ensures that the query phrase is at the end of user text.
QUERY_PREFIX = ('Inspect this image. Identify visual evidence for the following expression. '
                'Treat the expression as a query, not as an assertion of truth.\nEXPRESSION:\n')

def build_query_batch(image, suffix):
    # No system prompt is required for attention probing. Exactly one image per request.
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text',
              'text': QUERY_PREFIX + suffix}]}]
    prompt = qwen_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return qwen_processor(text=[prompt], images=[image], return_tensors='pt').to(QWEN_DEVICE)

def changed_span(base, full):
    a, b = base['input_ids'][0].tolist(), full['input_ids'][0].tolist()
    start = 0
    while start < min(len(a), len(b)) and a[start] == b[start]: start += 1
    end_a, end_b = len(a), len(b)
    while end_a > start and end_b > start and a[end_a - 1] == b[end_b - 1]:
        end_a -= 1; end_b -= 1
    return list(range(start, end_b))

@torch.inference_mode()
def qwen_phrase_attention(image, phrase, base=None):
    if base is None: base = build_query_batch(image, '')
    batch = build_query_batch(image, phrase)
    img_id = int(qwen.config.image_token_id)
    image_positions = torch.where(batch['input_ids'][0] == img_id)[0]
    if image_positions.numel() == 0:
        raise RuntimeError('No Qwen image tokens detected in input_ids')
    grid = batch['image_grid_thw'][0].tolist()
    merge = int(qwen.config.vision_config.spatial_merge_size)
    t, gh_raw, gw_raw = [int(x) for x in grid]
    assert gh_raw % merge == 0 and gw_raw % merge == 0
    gh, gw = gh_raw // merge, gw_raw // merge
    expected = t * gh * gw
    assert t == 1, f'Expected static image with one temporal grid, got {grid}'
    assert expected == image_positions.numel(), (
        f'Image-grid mismatch: {grid=} {merge=} {expected=} {image_positions.numel()=}')
    query_positions = changed_span(base, batch)
    assert query_positions and min(query_positions) > image_positions.max().item(), (
        'Could not align phrase tokens after image; examine tokenizer/template.')
    # Check that the differing prompt tokens actually represent the query phrase.
    found = qwen_processor.tokenizer.decode(batch['input_ids'][0, query_positions], skip_special_tokens=True)
    if re.sub(r'\s+', '', phrase).casefold() not in re.sub(r'\s+', '', found).casefold():
        raise ValueError(f'Unreliable phrase token alignment: target={phrase!r}, matched={found!r}')
    positions_t = torch.tensor(query_positions, device=QWEN_DEVICE, dtype=torch.long)
    img_positions_t = image_positions.to(QWEN_DEVICE)
    collected = {}
    handles = []

    def hook_for(index):
        def prehook(module, args, kwargs):
            hidden = kwargs.get('hidden_states', args[0] if args else None)
            cos, sin = kwargs['position_embeddings']
            shape = (*hidden.shape[:2], -1, module.head_dim)
            q = module.q_norm(module.q_proj(hidden).view(shape)).transpose(1, 2)
            k = module.k_norm(module.k_proj(hidden).view(shape)).transpose(1, 2)
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            k = k.repeat_interleave(module.num_key_value_groups, dim=1)
            # q: B,H,S,D ; k: B,H,S,D. Only compute requested query rows.
            q_part = q.index_select(-2, positions_t).float()
            logits = torch.matmul(q_part, k.float().transpose(-1, -2)) * module.scaling
            seq_idx = torch.arange(k.shape[-2], device=logits.device)
            causal_forbidden = seq_idx[None, :] > positions_t[:, None]
            logits.masked_fill_(causal_forbidden[None, None, :, :], torch.finfo(logits.dtype).min)
            probs = logits.softmax(-1)             # denominator includes all visible tokens
            image_probs = probs.index_select(-1, img_positions_t)  # B,H,Q,I
            token_maps = image_probs[0].mean(0)   # Q,I averaged across heads
            collected[index] = token_maps.cpu().numpy().astype('float32')
        return prehook

    try:
        for idx in QWEN_LAYERS:
            module = qwen.model.language_model.layers[idx].self_attn
            handles.append(module.register_forward_pre_hook(hook_for(idx), with_kwargs=True))
        _ = qwen(**batch, use_cache=False, output_attentions=False, return_dict=True)
    finally:
        for h in handles: h.remove()
    if set(collected) != set(QWEN_LAYERS):
        raise RuntimeError(f'Qwen hooks missing: {set(QWEN_LAYERS) - set(collected)}')
    per_token = np.stack([collected[i] for i in QWEN_LAYERS]).mean(0)  # Q,patches
    raw_mass = float(per_token.sum(-1).mean())
    raw_map = per_token.mean(0).reshape(gh, gw)
    tokens = [qwen_processor.tokenizer.decode([batch['input_ids'][0, p].item()],
                                               skip_special_tokens=False) for p in query_positions]
    return {'raw_map': raw_map, 'normalized_map': normalize_np(raw_map),
            'token_maps': per_token.reshape(len(tokens), gh, gw), 'tokens': tokens,
            'raw_image_mass': raw_mass, 'grid': [gh, gw], 'layer_ids': QWEN_LAYERS,
            'matched_query_tokens': found}

def normalize_np(arr):
    arr = np.nan_to_num(np.asarray(arr, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    amin, amax = float(arr.min()), float(arr.max())
    return (arr - amin) / (amax - amin + 1e-8)

## 5. Official CLIP Surgery and Grad-ECLIP on GPU 1

- **Surgery:** official `surgery_clip.load('CS-ViT-B/16')` and `clip_feature_surgery`. For independent concept queries it uses the author's *single-text* branch with an empty-text redundant feature; for the full caption we apply the same rule. No gradient is involved.
- **Grad-ECLIP:** uses original OpenAI CLIP ViT-B/16. Replicates the [author's published image notebook](https://raw.githubusercontent.com/Cyang-Zhao/Grad-Eclip/main/grad_eclip_image.ipynb): form one-head Q/K/V from the last residual attention block, differentiate image–text cosine against **that block's CLS attention output**, and aggregate `ReLU(sum_channels(grad_cls * value_patch * normalized_cosine(q_cls,k_patch)))`. Last block only, as in the paper's image-encoder experiments.
- **Score perturbation:** `score(full_caption) - score(caption_without_concept)`; positive deltas indicate that the phrase contributes to the **specific CLIP similarity** under this prompt rewrite. It is not an absolute semantic ranking; phrase omission can change grammar.

In [ ]:
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
clip_preprocess = Compose([
    Resize((CLIP_SIDE, CLIP_SIDE), interpolation=InterpolationMode.BICUBIC),
    ToTensor(), Normalize(CLIP_MEAN, CLIP_STD),
])

def torch_map(x, size):
    x = x.float().reshape(1, 1, *x.shape)
    return F.interpolate(x, size=size, mode='bilinear', align_corners=False)[0, 0].cpu().numpy()

@torch.inference_mode()
def clip_surgery_maps(image, phrases):
    x = clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
    feats = F.normalize(surgery_model.encode_image(x), dim=-1)
    assert feats.ndim == 3, f'Expect Surgery image patch features, got {feats.shape}'
    # Full CLIP text encoder, not prompt-ensemble: ensures phrases/caption are passed as written.
    tokens = surgery_clip.tokenize(phrases + [''], truncate=True).to(CLIP_DEVICE)
    txt = F.normalize(surgery_model.encode_text(tokens), dim=-1)
    maps = []
    for text_vec in txt[:-1]:
        sim = surgery_clip.clip_feature_surgery(feats, text_vec[None, :], redundant_feats=txt[-1:])
        scores = sim[0, 1:, 0]      # CLS token excluded
        side = math.isqrt(scores.numel())
        assert side * side == scores.numel(), 'Unexpected CLIP Surgery token spatial layout'
        maps.append(normalize_np(scores.reshape(side, side).float().cpu().numpy()))
    del x, feats, txt
    return maps

# Last block: one-head Q/K/V reconstruction from the Grad-ECLIP author notebook.
def grad_eclip_image_forward(x):
    v = clip_model.visual
    dtype = v.conv1.weight.dtype
    with torch.no_grad():
        patches = v.conv1(x.to(dtype))
        gh, gw = patches.shape[-2:]
        patches = patches.reshape(1, patches.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(dtype) + torch.zeros(1, 1, patches.shape[-1], device=CLIP_DEVICE, dtype=dtype)
        h = torch.cat([cls, patches], dim=1)
        # At CLIP_SIDE==224 exactly the official position embedding resolution.
        assert h.shape[1] == v.positional_embedding.shape[0], 'Set CLIP_SIDE=224 or add position interpolation.'
        h = v.ln_pre(h + v.positional_embedding.to(dtype))
        h = h.permute(1, 0, 2)  # S,B,C
        for blk in v.transformer.resblocks[:-1]:
            h = blk(h)
    h = h.detach().requires_grad_(True)
    blk = v.transformer.resblocks[-1]
    z = blk.ln_1(h)
    q, k, values = F.linear(z, blk.attn.in_proj_weight, blk.attn.in_proj_bias).chunk(3, dim=-1)
    # Single-head attention, as in author notebook's clip_encode_dense(..., n=1).
    q_scaled = q * (q.shape[-1] ** -0.5)
    probs = (q_scaled[:, 0] @ k[:, 0].T).softmax(dim=-1)  # S,S
    attn_output = (probs @ values[:, 0]).unsqueeze(1)    # S,1,C
    after_attn = F.linear(attn_output, blk.attn.out_proj.weight, blk.attn.out_proj.bias)
    h = h + after_attn
    h = h + blk.mlp(blk.ln_2(h))
    image_embedding = (v.ln_post(h.permute(1, 0, 2))[:, 0] @ v.proj)
    image_embedding = F.normalize(image_embedding.float(), dim=-1)
    return image_embedding, q, k, values, attn_output, (gh, gw)

@torch.no_grad()
def plain_clip_text(phrases):
    return F.normalize(clip_model.encode_text(openai_clip.tokenize(phrases, truncate=True).to(CLIP_DEVICE)).float(), dim=-1)

@torch.no_grad()
def plain_clip_score(image, phrases):
    emb = F.normalize(clip_model.encode_image(clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)).float(), dim=-1)
    return (emb @ plain_clip_text(phrases).T)[0].float().cpu().numpy()

def grad_eclip_maps(image, phrases):
    xx = clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
    with torch.no_grad(): txt = plain_clip_text(phrases)
    emb, q, k, values, attn_output, grid = grad_eclip_image_forward(xx)
    scores = (emb @ txt.T).reshape(-1)
    with torch.no_grad():
        q_cls = F.normalize(q[0, 0].float(), dim=-1)
        k_patch = F.normalize(k[1:, 0].float(), dim=-1)
        relaxed = k_patch @ q_cls
        relaxed = (relaxed - relaxed.min()) / (relaxed.max() - relaxed.min() + 1e-8)
        val_patch = values[1:, 0].float()
    maps = []
    try:
        for j in range(len(phrases)):
            grad = torch.autograd.grad(scores[j], attn_output, retain_graph=j < len(phrases) - 1)[0]
            chan_weight = grad[0, 0].float()  # gradient at CLS attention output
            emap = (val_patch * chan_weight[None, :] * relaxed[:, None]).sum(-1).relu()
            maps.append(normalize_np(emap.reshape(grid).detach().cpu().numpy()))
        sims = scores.detach().cpu().numpy().astype('float32')
    finally:
        del emb, q, k, values, attn_output, xx
    return maps, sims

# Sanity check math without loading any data: Grad-ECLIP should differentiate a scalar
# w.r.t. the last attention output, not a detached tensor or a Surgery embedding.

## 6. Concept importance interventions and visualization utilities

The **deletion test** blurs the top 20% of pixels selected by each CLIP map and compares the decrease in original CLIP cosine score with a seeded random-mask control. This is a perturbation sanity check, sensitive to blur artifacts and image-text distribution shift; it is not a proof of ground-truth explanations. We run the test for the full caption and its concepts on **the same vanilla CLIP evaluator**.

In [ ]:
def remove_phrase(caption, phrase):
    # Conservative literal substring deletion. If absent, the LOO estimate is undefined.
    match = re.search(re.escape(phrase), caption, flags=re.I)
    if not match: return None
    new = (caption[:match.start()] + caption[match.end():]).strip()
    new = re.sub(r'\s+', ' ', new)
    new = re.sub(r'\s+([,.!?])', r'\1', new)
    return new if new and new != caption else None

def resize_heatmap(m, wh):
    w, h = wh
    return cv2.resize(np.asarray(m, dtype='float32'), (w, h), interpolation=cv2.INTER_LINEAR)

def overlay(image, m, alpha=0.48):
    source = np.asarray(image.convert('RGB'), dtype=np.uint8)
    smooth = resize_heatmap(normalize_np(m), image.size)
    cmap = cv2.applyColorMap((smooth * 255).astype(np.uint8), cv2.COLORMAP_TURBO)
    cmap = cv2.cvtColor(cmap, cv2.COLOR_BGR2RGB)
    return np.clip(source * (1 - alpha) + cmap * alpha, 0, 255).astype(np.uint8)

def plot_panels(image, phrases, maps_by_method, image_id, out_dir):
    methods = ['Qwen3-VL attention', 'CLIP Surgery', 'Grad-ECLIP']
    nrows = len(phrases)
    fig, axes = plt.subplots(nrows, 4, figsize=(16, max(3, 2.9*nrows)), squeeze=False)
    for i, phrase in enumerate(phrases):
        axes[i, 0].imshow(image); axes[i, 0].set_title('Original / ' + ('FULL CAPTION' if i == 0 else 'CONCEPT'), fontsize=10)
        axes[i, 0].set_ylabel(phrase[:65], fontsize=9)
        for j, method in enumerate(methods):
            axes[i, j+1].imshow(overlay(image, maps_by_method[method][i]))
            axes[i, j+1].set_title(method, fontsize=10)
        for a in axes[i]: a.axis('off')
    fig.suptitle(f'Image {image_id} — matched expressions / non-comparable map scales', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fname = out_dir / 'comparison.png'
    fig.savefig(fname, dpi=145, bbox_inches='tight')
    plt.show(); plt.close(fig)
    return fname

def plot_caption_tokens(image, attention_result, path):
    n = len(attention_result['tokens'])
    if n < 1: return
    cols = min(5, n)
    rows = math.ceil(n / cols)
    fig, ax = plt.subplots(rows, cols, figsize=(3.0 * cols, 2.7 * rows), squeeze=False)
    for j, a in enumerate(ax.flat):
        a.axis('off')
        if j < n:
            a.imshow(overlay(image, attention_result['token_maps'][j]))
            a.set_title(f'Qwen token {j}: {attention_result["tokens"][j]!r}', fontsize=9)
    fig.suptitle('Full-caption *prompt-token* attention (not token generation attribution)')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); fig.savefig(path, dpi=125, bbox_inches='tight')
    plt.show(); plt.close(fig)

def gaussian_delete(image, heatmap, fraction=TOP_MASK_FRACTION, seed=SEED):
    src = np.asarray(image.convert('RGB'), dtype=np.uint8)
    H, W = src.shape[:2]
    hmap = resize_heatmap(normalize_np(heatmap), (W, H))
    k = max(1, min(H * W, int(fraction * H * W)))
    top_idx = np.argpartition(hmap.ravel(), -k)[-k:]
    mask = np.zeros(H*W, bool); mask[top_idx] = True
    # Fixed blur kernel for consistency on target and random control.
    blurred = cv2.GaussianBlur(src, (31, 31), 11.0)
    deleted = src.copy(); deleted.reshape(-1, 3)[mask] = blurred.reshape(-1, 3)[mask]
    rng = np.random.default_rng(seed)
    ridx = rng.choice(H*W, size=k, replace=False)
    random_deleted = src.copy()
    random_deleted.reshape(-1, 3)[ridx] = blurred.reshape(-1, 3)[ridx]
    return Image.fromarray(deleted), Image.fromarray(random_deleted)

# GPU 1 worker: single thread preserves CUDA device affinity; no Qwen access here.
def compute_clip_job(image, caption, concepts, image_id):
    torch.cuda.set_device(CLIP_DEVICE)
    phrases = [caption] + concepts
    surgery_maps = clip_surgery_maps(image, phrases)
    grad_maps, grad_scores = grad_eclip_maps(image, phrases)
    with torch.no_grad():
        orig_scores = plain_clip_score(image, phrases)
    assert len(surgery_maps) == len(grad_maps) == len(phrases)
    return {'surgery_maps': surgery_maps, 'grad_maps': grad_maps,
            'grad_scores': grad_scores.tolist(), 'plain_scores': orig_scores.tolist()}

### Additional token-importance analysis

For **Qwen subwords**, `qwen_subword_attention.csv` records raw attention mass to visual tokens, along with each individual subword's map in `caption_subtokens.png`. For **human-readable caption words**, `caption_word_LOO.csv` tests the change in vanilla CLIP cosine similarity when each word is removed (same image). These are two different tokenizations and do **not** claim one-to-one attribution across models. The plots let you inspect whether content words and function words behave differently.

In [ ]:
def save_word_importance(image_id, caption, image, qwen_full, out_dir):
    # Qwen's actual phrase token boundaries and unnormalized attention to the image:
    token_masses = qwen_full['token_maps'].reshape(len(qwen_full['tokens']), -1).sum(-1)
    tokens = [{'image_id': image_id, 'qwen_token_index': i, 'qwen_token_text': tok,
               'qwen_image_attention_mass': float(token_masses[i])}
              for i, tok in enumerate(qwen_full['tokens'])]
    pd.DataFrame(tokens).to_csv(out_dir / 'qwen_subword_attention.csv', index=False)
    # Word-level textual deletion. Literal word removal is a sensitivity test, not causal proof.
    matches = list(re.finditer(r"\b[\w'-]+\b", caption, flags=re.UNICODE))
    if not matches: return
    candidates = []
    for m in matches:
        dropped = (caption[:m.start()] + caption[m.end():]).strip()
        dropped = re.sub(r'\s+', ' ', dropped)
        dropped = re.sub(r'\s+([,.!?])', r'\1', dropped)
        candidates.append(dropped)
    baseline = float(plain_clip_score(image, [caption])[0])
    # Batch texts into groups; image re-encoding dominates here, so reuse its embedding.
    unique = list(dict.fromkeys(candidates))
    word_score_lookup = {}
    with torch.no_grad():
        xx = clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
        visual = F.normalize(clip_model.encode_image(xx).float(), dim=-1)
        for start in range(0, len(unique), 64):
            chunk = unique[start:start+64]
            txt = plain_clip_text(chunk)
            score = (visual @ txt.T)[0].float().cpu().tolist()
            word_score_lookup.update(zip(chunk, score))
    rows = [{'image_id': image_id, 'word_index': i, 'word': m.group(),
             'start_char': m.start(), 'end_char': m.end(),
             'similarity_full': baseline, 'similarity_without_word': float(word_score_lookup[dropped]),
             'LOO_score_delta': baseline - float(word_score_lookup[dropped])}
            for i, (m, dropped) in enumerate(zip(matches, candidates))]
    pd.DataFrame(rows).to_csv(out_dir / 'caption_word_LOO.csv', index=False)
    # Plot top 16 words by absolute change; color comes from matplotlib default.
    top = pd.DataFrame(rows).sort_values('LOO_score_delta', key=abs, ascending=False).head(16)
    fig, ax = plt.subplots(figsize=(10, max(3.5, len(top)*0.35)))
    ax.barh([f"{r.word} [{r.word_index}]" for r in top.itertuples()], top['LOO_score_delta'])
    ax.set_xlabel('Full CLIP cosine − cosine without word')
    ax.set_title('Caption word leave-one-out sensitivity (not causal attribution)')
    ax.axvline(0, linewidth=1)
    ax.invert_yaxis(); fig.tight_layout()
    fig.savefig(out_dir / 'word_importance.png', dpi=130, bbox_inches='tight')
    plt.show(); plt.close(fig)

## 7. Two-GPU producer/consumer experiment loop

Qwen on `cuda:0` computes concepts and attention while a single worker on `cuda:1` computes CLIP heatmaps from the **previous** image. Only one outstanding CLIP job is queued (bounded producer), reducing GPU memory spikes. All numerical maps, prompts, and settings are saved to disk for reproducibility. The loop fails visibly on any unexpected model incompatibility instead of inventing result data.

In [ ]:
from tqdm.auto import tqdm

results = []
manifest = []
pending = []
executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix='clip_gpu1')

def finish_clip_task(task):
    sample, image, caption, concepts, raw_llm, qwen_maps, qwen_records, qwen_token_full = task['context']
    clip_result = task['future'].result()
    image_id = sample['image_id']
    img_dir = OUTPUT_DIR / f'{task["index"]:04d}_{re.sub(r"[^a-zA-Z0-9_-]", "_", image_id)[:70]}'
    img_dir.mkdir(parents=True, exist_ok=True)
    phrases = [caption] + concepts
    aligned_maps = {
        'Qwen3-VL attention': qwen_maps,
        'CLIP Surgery': clip_result['surgery_maps'],
        'Grad-ECLIP': clip_result['grad_maps'],
    }
    figfile = plot_panels(image, phrases, aligned_maps, image_id, img_dir)
    if VISUALIZE_CAPTION_SUBTOKENS:
        plot_caption_tokens(image, qwen_token_full, img_dir / 'caption_subtokens.png')

    loo_targets = [remove_phrase(caption, c) for c in concepts]
    unique_loo = list(dict.fromkeys(x for x in loo_targets if x))
    loo_score_map = {}
    if unique_loo:
        with torch.no_grad():
            score_vals = plain_clip_score(image, unique_loo)
        loo_score_map = dict(zip(unique_loo, map(float, score_vals)))
    rows = []
    for i, expr in enumerate(phrases):
        row = {'image_id': image_id, 'image_path': sample['image_path'], 'kind': 'caption' if i == 0 else 'concept',
               'expression': expr, 'reference_caption': sample['reference_caption'],
               'caption': caption, 'clip_similarity': float(clip_result['plain_scores'][i]),
               'grad_eclip_reconstructed_clip_similarity': float(clip_result['grad_scores'][i]),
               'qwen_image_attention_mass': float(qwen_records[i]['raw_image_mass']),
               'qwen_matched_tokens': qwen_records[i]['matched_query_tokens'],
               'qwen_layers': str(QWEN_LAYERS), 'figure_path': str(figfile)}
        if i > 0:
            without = loo_targets[i-1]
            row['caption_without_concept'] = without
            row['clip_LOO_delta'] = float(clip_result['plain_scores'][0] - loo_score_map[without]) if without else np.nan
        else: row['clip_LOO_delta'] = np.nan
        if RUN_DELETION:
            for method in ('CLIP Surgery', 'Grad-ECLIP'):
                changed, random_changed = gaussian_delete(image, aligned_maps[method][i],
                                                          seed=SEED + task['index']*100 + i)
                with torch.no_grad():
                    masked_scores = plain_clip_score(changed, [expr])
                    random_scores = plain_clip_score(random_changed, [expr])
                row[f'{method}_score_drop_topmask'] = float(clip_result['plain_scores'][i] - masked_scores[0])
                row[f'{method}_score_drop_randommask'] = float(clip_result['plain_scores'][i] - random_scores[0])
        rows.append(row)
    if SAVE_NPZ:
        arrays = {}
        for m_name, all_maps in aligned_maps.items():
            key = {'Qwen3-VL attention': 'qwen', 'CLIP Surgery': 'surgery', 'Grad-ECLIP': 'grad_eclip'}[m_name]
            for i, hm in enumerate(all_maps): arrays[f'{key}_{i}'] = np.asarray(hm, dtype='float32')
        arrays['caption_token_maps'] = np.asarray(qwen_token_full['token_maps'], dtype='float32')
        np.savez_compressed(img_dir / 'heatmaps.npz', **arrays)
    info = {
        'image_id': image_id, 'image_path': sample['image_path'], 'caption': caption,
        'reference_caption': sample['reference_caption'], 'concepts': concepts,
        'raw_llm_response': raw_llm, 'qwen_attention_records': qwen_records,
        'qwen_caption_subtokens': qwen_token_full['tokens'],
        'clip_side': CLIP_SIDE, 'top_mask_fraction': TOP_MASK_FRACTION,
        'figfile': str(figfile),
    }
    (img_dir / 'metadata.json').write_text(json.dumps(info, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    results.extend(rows)
    manifest.append({'image_id': image_id, 'folder': str(img_dir), 'caption': caption, 'concepts': concepts})
    pd.DataFrame(results).to_csv(OUTPUT_DIR / 'concept_importance.csv', index=False)
    # Token/word analysis is saved separately; spaces and BPE subwords have different boundaries.
    save_word_importance(image_id, caption, image, qwen_token_full, img_dir)
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Completed:', image_id, '|', len(phrases), 'queries |', figfile.name)

try:
    for sample_idx, sample in enumerate(tqdm(samples, desc='Qwen: GPU 0')):
        image = load_rgb(sample['image_path'])
        caption, concepts, raw_llm = extract_caption_concepts(image, sample['reference_caption'])
        phrases = [caption] + concepts
        qwen_maps, qwen_records, qwen_token_full = [], [], None
        base = build_query_batch(image, '')
        for i, phrase in enumerate(phrases):
            attn = qwen_phrase_attention(image, phrase, base=base)
            qwen_maps.append(attn['normalized_map'])
            qwen_records.append({k: v for k, v in attn.items() if k not in ('raw_map', 'normalized_map', 'token_maps')})
            if i == 0: qwen_token_full = attn
        context = (sample, image, caption, concepts, raw_llm, qwen_maps, qwen_records, qwen_token_full)
        fut = executor.submit(compute_clip_job, image, caption, concepts, sample['image_id'])
        pending.append({'future': fut, 'context': context, 'index': sample_idx})
        # Bound pending jobs; Qwen GPU0 overlaps CLIP GPU1 across adjacent images.
        if len(pending) > 1:
            finish_clip_task(pending.pop(0))
    while pending:
        finish_clip_task(pending.pop(0))
finally:
    executor.shutdown(wait=True)

print('Rows:', len(results), '| Output:', OUTPUT_DIR)

## 8. Analyze saliency and caption-versus-concept scores

**Interpretation tips:**
- `clip_LOO_delta` is a signed change in *full-caption similarity* after deleting one phrase from the caption. A positive value is not guaranteed even if that phrase appears in the image.
- `qwen_image_attention_mass` is the share of attention probability Qwen assigned to image tokens from the query phrase; attention to *all other tokens* remains in the denominator.
- `score_drop_topmask` can be compared to `score_drop_randommask`, but both are affected by blur strength and area. Evaluate over a large representative sample before drawing conclusions.
- `grad_eclip_reconstructed_clip_similarity` is reported separately because the Grad-ECLIP author's one-head forward **does not equal the original multihead CLIP forward**. Do not combine its score scale with `clip_similarity`.

In [ ]:
summary = pd.read_csv(OUTPUT_DIR / 'concept_importance.csv')
display(summary.head(20))
if summary['clip_LOO_delta'].notna().any():
    tmp = summary.loc[summary.kind.eq('concept')].copy()
    tmp = tmp.sort_values('clip_LOO_delta', ascending=False)
    print('Largest positive leave-one-concept-out changes:')
    display(tmp[['image_id', 'expression', 'clip_LOO_delta', 'clip_similarity', 'qwen_image_attention_mass']].head(25))
if RUN_DELETION:
    stats = []
    for method in ('CLIP Surgery', 'Grad-ECLIP'):
        d = summary[f'{method}_score_drop_topmask']
        r = summary[f'{method}_score_drop_randommask']
        stats.append({'method': method, 'mean_topmask_drop': d.mean(),
                      'mean_randommask_drop': r.mean(), 'mean_advantage_over_random': (d-r).mean()})
    display(pd.DataFrame(stats))
config = {'dataset': DATASET_SLUG, 'root': str(ROOT), 'models': [QWEN_MODEL_ID,'CS-ViT-B/16','ViT-B/16'],
          'torch': torch.__version__, 'transformers': HF_VERSION, 'selected_qwen_layers': QWEN_LAYERS,
          'seed': SEED, 'max_images': MAX_IMAGES, 'max_concepts': MAX_CONCEPTS,
          'qwen_max_pixels_config': QWEN_MAX_PIXELS, 'clip_side': CLIP_SIDE,
          'attention_interpretation': 'user prompt concept-token to image-token, causal QK softmax',
          'grad_eclip_interpretation': 'single-head last-block author formulation',
          'mask_fraction': TOP_MASK_FRACTION}
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Saved:', [p.name for p in OUTPUT_DIR.glob('*') if p.is_file()])

## 9. Research follow-ups / ablation checklist

**Minimum recommended ablations:** compare `QWEN_LAYER_FRACTIONS` sets; test `QWEN_MAX_PIXELS=384²` vs `448²`; compare single-head author's Grad-ECLIP with an original-multihead adaptation separately; inspect concept grounding manually on ~50 examples; quantify deletion advantage over random with confidence intervals. For word-level explanations, the `caption_subtokens.png` token maps explain **prompt tokens**, not the words predicted during free-form generation. If you need generation-time attribution, capture decoder next-token attention/gradients as an explicitly separate protocol.

**Sources and exact method matches:** CLIP Surgery: Li et al., *Pattern Recognition* 2025, DOI [10.1016/j.patcog.2025.111409](https://doi.org/10.1016/j.patcog.2025.111409) (architecture+feature surgery; original GitHub code used here). Grad-ECLIP: Zhao et al., *ICML* 2024 (Eq. 7–8; normalized Q/K similarity as spatial weight; last-block single-head author code). Qwen3-VL: Hugging Face `Qwen3VLTextAttention.forward` (Q/K layer normalization → RoPE → causal scaled dot product). These sources **do not** establish that attention weights alone measure causal feature importance.